<a href="https://colab.research.google.com/github/HemanthSagarSR/AI_Lab-/blob/main/week8_FOL_FC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import itertools
import re

class Rule:
    def __init__(self, premises, conclusion):
        self.premises = premises
        self.conclusion = conclusion

    def __repr__(self):
        prem = " & ".join([str(p) for p in self.premises])
        return f"{prem} => {self.conclusion}"

def is_variable(x):
    return isinstance(x, str) and len(x) > 0 and x.islower()

def parse_literal(text):
    text = text.strip()
    match = re.match(r"^([A-Za-z0-9_]+)\((.*)\)$", text)
    if not match:
        raise ValueError(f"Invalid format: {text}")
    pred = match.group(1)
    args = [arg.strip() for arg in match.group(2).split(",") if arg.strip()]
    return (pred, *args)

def parse_input_rule(text):
    if "=>" not in text:
        raise ValueError("Rule must contain '=>'")
    left, right = text.split("=>")
    premises = [parse_literal(p) for p in left.split("&")]
    conclusion = parse_literal(right)
    return Rule(premises, conclusion)

def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    elif theta is False:
        return None

    if x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, (tuple, list)) and isinstance(y, (tuple, list)):
        if len(x) != len(y):
            return None
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
            if theta is None:
                return None
        return theta
    else:
        return None

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    elif x in theta:
        return unify(var, theta[x], theta)
    else:
        theta[var] = x
        return theta

def subst(theta, expr):
    if isinstance(expr, (tuple, list)):
        return tuple(subst(theta, element) for element in expr)
    elif expr in theta:
        return theta[expr]
    else:
        return expr

var_counter = 0
def standardize_variables(rule):
    global var_counter
    var_counter += 1
    var_map = {}

    def remap(expr):
        if isinstance(expr, (tuple, list)):
            return tuple(remap(e) for e in expr)
        elif is_variable(expr):
            if expr not in var_map:
                var_map[expr] = f"{expr}_{var_counter}"
            return var_map[expr]
        return expr

    new_premises = [remap(p) for p in rule.premises]
    new_conclusion = remap(rule.conclusion)
    return Rule(new_premises, new_conclusion)

def fol_fc_ask(kb_rules, kb_facts, alpha):
    kb_facts = set(kb_facts)

    for fact in kb_facts:
        phi = unify(fact, alpha)
        if phi is not None:
            return phi

    while True:
        new = set()
        for rule in kb_rules:
            std_rule = standardize_variables(rule)
            p_terms = std_rule.premises
            q = std_rule.conclusion

            for fact_combo in itertools.product(kb_facts, repeat=len(p_terms)):
                theta = {}
                possible = True
                for premise, fact in zip(p_terms, fact_combo):
                    theta = unify(premise, fact, theta)
                    if theta is None:
                        possible = False
                        break

                if possible:
                    q_prime = subst(theta, q)
                    already_exists = False
                    for existing in kb_facts | new:
                        if unify(q_prime, existing) == {}:
                            already_exists = True
                            break

                    if not already_exists:
                        print(f"Inferred: {q_prime}({', '.join(q_prime[1:])})")
                        new.add(q_prime)

                        phi = unify(q_prime, alpha)
                        if phi is not None:
                            query_vars = {v for v in alpha if is_variable(v)}
                            filtered_phi = {k: v for k, v in phi.items() if k in query_vars}
                            return filtered_phi if filtered_phi else True

        if not new:
            return False
        kb_facts.update(new)

if __name__ == "__main__":
    print("--- FIRST ORDER LOGIC KNOWLEDGE BASE INPUT ---")
    print("Variables must be single lowercase letters.")
    print("Constants must start with an uppercase letter.\n")

    user_facts = set()
    print("Enter initial facts one by one. Enter an empty line to finish:")
    while True:
        line = input("> ").strip()
        if not line:
            break
        try:
            user_facts.add(parse_literal(line))
        except ValueError as e:
            print(f"Error: {e}. Try again.")

    user_rules = []
    print("\nEnter implications rules. Enter an empty line to finish:")
    while True:
        line = input("> ").strip()
        if not line:
            break
        try:
            user_rules.append(parse_input_rule(line))
        except ValueError as e:
            print(f"Error: {e}. Try again.")

    print("\nEnter the query atomic sentence to prove:")
    while True:
        line = input("> ").strip()
        if line:
            try:
                user_query = parse_literal(line)
                break
            except ValueError as e:
                print(f"Error: {e}. Try again.")

    print("\n--- Running Inference Engine ---")
    result = fol_fc_ask(user_rules, user_facts, user_query)
    print(f"\nResult: {result}")


--- FIRST ORDER LOGIC KNOWLEDGE BASE INPUT ---
Variables must be single lowercase letters.
Constants must start with an uppercase letter.

Enter initial facts one by one. Enter an empty line to finish:
> American(Robert)
> Enemy(A, America)
>  Owns(A, T1)
> Missile(T1)
> 

Enter implications rules. Enter an empty line to finish:
> Missile(x) => Weapon(x)
> Enemy(x, America) => Hostile(x)
>  Missile(x) & Owns(A, x) => Sells(Robert, x, A)
> American(p) & Weapon(q) & Sells(p, q, r) & Hostile(r) => Criminal(p)
> 

Enter the query atomic sentence to prove:
> Criminal(Robert)

--- Running Inference Engine ---
Inferred: ('Weapon', 'T1')(T1)
Inferred: ('Hostile', 'A')(A)
Inferred: ('Sells', 'Robert', 'T1', 'A')(Robert, T1, A)
Inferred: ('Criminal', 'Robert')(Robert)

Result: True
